# 2. 神经网络量子态

本节介绍神经网络作为量子态表示的方法，包括各种类型的神经网络量子态及其特点。

## 2.1 神经网络作为波函数的表示

神经网络量子态（Neural Network Quantum States, NQS）是使用神经网络来表示量子多体系统波函数的方法。这种方法在近年来得到了广泛关注，因为它能够有效地捕捉量子系统中的复杂关联。

### 基本原理

神经网络量子态的核心思想是将波函数表示为神经网络的输出：

$$\psi(\mathbf{x}; \mathbf{\theta}) = f_{NN}(\mathbf{x}; \mathbf{\theta})$$

其中$\mathbf{x}$表示系统组态（如自旋组态、电子位置等），$\mathbf{\theta}$表示神经网络的权重和偏置参数，$f_{NN}$是神经网络的输出函数。

### 神经网络量子态的优势

1. **表达能力**：神经网络具有强大的函数逼近能力，可以表示复杂的波函数
2. **灵活性**：可以适应各种对称性和约束条件
3. **可扩展性**：计算复杂度随系统规模增长较慢
4. **自动优化**：可以通过梯度下降等方法自动优化参数

### 波函数的归一化

在实际应用中，我们通常不需要显式地归一化波函数，因为在计算能量期望值时，归一化因子会被约去：

$$E = \frac{\langle \psi | \hat{H} | \psi \rangle}{\langle \psi | \psi \rangle} = \frac{\int |\psi(\mathbf{x})|^2 E_L(\mathbf{x}) d\mathbf{x}}{\int |\psi(\mathbf{x})|^2 d\mathbf{x}} = \frac{\sum_i |\psi(\mathbf{x}_i)|^2 E_L(\mathbf{x}_i)}{\sum_i |\psi(\mathbf{x}_i)|^2}$$

其中$E_L(\mathbf{x}) = \frac{\hat{H}\psi(\mathbf{x})}{\psi(\mathbf{x})}$是局域能量。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

# Set style for plots
plt.style.use('seaborn-v0_8')

# Check if GPU is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 2.2 受限玻尔兹曼机（RBM）量子态

受限玻尔兹曼机（Restricted Boltzmann Machine, RBM）是最早用于表示量子态的神经网络之一。RBM是一种两层神经网络，包括可见层和隐藏层，层内无连接，层间全连接。

### RBM量子态的数学表示

对于离散自旋系统，RBM量子态可以表示为：

$$\psi(\mathbf{s}; \mathbf{\theta}) = e^{\sum_i a_i s_i} \prod_{j=1}^{N_h} 2\cosh\left(\sum_i W_{ij} s_i + b_j\right)$$

其中$\mathbf{s} = (s_1, s_2, ..., s_N)$是自旋组态（$s_i = \pm 1$），$\mathbf{\theta} = \{\mathbf{a}, \mathbf{b}, \mathbf{W}\}$是RBM的参数，$\mathbf{a}$和$\mathbf{b}$分别是可见层和隐藏层的偏置，$\mathbf{W}$是权重矩阵，$N_h$是隐藏单元的数量。

### RBM量子态的特点

1. **精确表示**：RBM可以精确表示一些重要的量子态，如矩阵乘积态（MPS）和树张量网络（TTN）
2. **高效计算**：波函数及其导数的计算可以高效进行
3. **物理直觉**：隐藏单元可以解释为系统中的隐藏自由度或关联

### RBM量子态的实现

In [ ]:
class RBMWavefunction(nn.Module):
    """
    Restricted Boltzmann Machine (RBM) wave function for spin systems
    
    Parameters:
    n_visible: number of visible units (spins)
    n_hidden: number of hidden units
    """
    def __init__(self, n_visible, n_hidden):
        super(RBMWavefunction, self).__init__()
        self.n_visible = n_visible
        self.n_hidden = n_hidden
        
        # Parameters
        self.a = nn.Parameter(torch.randn(n_visible))  # Visible bias
        self.b = nn.Parameter(torch.randn(n_hidden))  # Hidden bias
        self.W = nn.Parameter(torch.randn(n_visible, n_hidden))  # Weights
        
    def forward(self, x):
        """
        Compute log of wave function amplitude
        
        Parameters:
        x: spin configurations, shape (batch_size, n_visible)
        
        Returns:
        log of wave function amplitude, shape (batch_size,)
        """
        # Visible layer contribution
        visible_term = torch.sum(x * self.a, dim=1)
        
        # Hidden layer contribution
        hidden_term = torch.sum(
            torch.log(2 * torch.cosh(torch.matmul(x, self.W) + self.b)),
            dim=1
        )
        
        return visible_term + hidden_term
    
    def log_psi(self, x):
        """
        Compute log of wave function
        
        Parameters:
        x: spin configurations, shape (batch_size, n_visible)
        
        Returns:
        log of wave function, shape (batch_size,)
        """
        return self.forward(x)
    
    def psi(self, x):
        """
        Compute wave function
        
        Parameters:
        x: spin configurations, shape (batch_size, n_visible)
        
        Returns:
        wave function, shape (batch_size,)
        """
        return torch.exp(self.log_psi(x))
    
    def local_energy(self, x, hamiltonian):
        """
        Compute local energy
        
        Parameters:
        x: spin configurations, shape (batch_size, n_visible)
        hamiltonian: function that computes local energy
        
        Returns:
        local energy, shape (batch_size,)
        """
        return hamiltonian(x, self)

# Example: Create RBM for a small spin system
n_spins = 4
n_hidden = 4
rbm = RBMWavefunction(n_spins, n_hidden).to(device)

# Generate random spin configurations
batch_size = 5
spins = torch.randint(0, 2, (batch_size, n_spins)).float() * 2 - 1  # Convert to +/- 1
spins = spins.to(device)

# Compute wave function
log_psi = rbm.log_psi(spins)
psi = rbm.psi(spins)

print("Spin configurations:")
print(spins.cpu().numpy())
print("\nLog of wave function:")
print(log_psi.cpu().detach().numpy())
print("\nWave function:")
print(psi.cpu().detach().numpy())

## 2.3 前馈神经网络量子态

前馈神经网络（Feedforward Neural Network, FNN）是另一种用于表示量子态的神经网络结构。与RBM不同，FNN可以有多个隐藏层，提供了更强的表达能力。

### FNN量子态的数学表示

对于离散自旋系统，FNN量子态可以表示为：

$$\psi(\mathbf{s}; \mathbf{\theta}) = e^{f_{NN}(\mathbf{s}; \mathbf{\theta})}$$

其中$f_{NN}(\mathbf{s}; \mathbf{\theta})$是前馈神经网络的输出，$\mathbf{\theta}$是网络的参数。

神经网络的具体形式为：

$$f_{NN}(\mathbf{s}; \mathbf{\theta}) = \mathbf{W}^{(L)} \sigma(\mathbf{W}^{(L-1)} \sigma(\cdots \sigma(\mathbf{W}^{(1)} \mathbf{s} + \mathbf{b}^{(1)}) \cdots ) + \mathbf{b}^{(L)})$$

其中$\mathbf{W}^{(l)}$和$\mathbf{b}^{(l)}$分别是第$l$层的权重和偏置，$\sigma$是激活函数（如tanh、ReLU等）。

### FNN量子态的特点

1. **强大的表达能力**：多层FNN可以表示非常复杂的函数
2. **灵活性**：可以设计不同的网络结构来适应不同的问题
3. **易于实现**：可以使用现有的深度学习框架轻松实现
4. **计算效率**：前向传播计算高效，适合大规模系统

In [ ]:
class FNNWavefunction(nn.Module):
    """
    Feedforward Neural Network (FNN) wave function for spin systems
    
    Parameters:
    n_visible: number of visible units (spins)
    hidden_sizes: list of hidden layer sizes
    activation: activation function ('tanh', 'relu', etc.)
    """
    def __init__(self, n_visible, hidden_sizes, activation='tanh'):
        super(FNNWavefunction, self).__init__()
        self.n_visible = n_visible
        self.hidden_sizes = hidden_sizes
        
        # Choose activation function
        if activation == 'tanh':
            self.activation = torch.tanh
        elif activation == 'relu':
            self.activation = F.relu
        else:
            raise ValueError(f"Unsupported activation: {activation}")
        
        # Create layers
        layers = []
        input_size = n_visible
        
        for hidden_size in hidden_sizes:
            layers.append(nn.Linear(input_size, hidden_size))
            input_size = hidden_size
        
        # Output layer (single neuron for log amplitude)
        layers.append(nn.Linear(input_size, 1))
        
        self.layers = nn.ModuleList(layers)
        
    def forward(self, x):
        """
        Compute log of wave function amplitude
        
        Parameters:
        x: spin configurations, shape (batch_size, n_visible)
        
        Returns:
        log of wave function amplitude, shape (batch_size,)
        """
        # Forward pass through hidden layers
        for i, layer in enumerate(self.layers[:-1]):
            x = self.activation(layer(x))
        
        # Output layer
        x = self.layers[-1](x)
        
        return x.squeeze(1)  # Remove last dimension
    
    def log_psi(self, x):
        """
        Compute log of wave function
        
        Parameters:
        x: spin configurations, shape (batch_size, n_visible)
        
        Returns:
        log of wave function, shape (batch_size,)
        """
        return self.forward(x)
    
    def psi(self, x):
        """
        Compute wave function
        
        Parameters:
        x: spin configurations, shape (batch_size, n_visible)
        
        Returns:
        wave function, shape (batch_size,)
        """
        return torch.exp(self.log_psi(x))
    
    def local_energy(self, x, hamiltonian):
        """
        Compute local energy
        
        Parameters:
        x: spin configurations, shape (batch_size, n_visible)
        hamiltonian: function that computes local energy
        
        Returns:
        local energy, shape (batch_size,)
        """
        return hamiltonian(x, self)

# Example: Create FNN for a small spin system
n_spins = 4
hidden_sizes = [8, 4]
fnn = FNNWavefunction(n_spins, hidden_sizes, activation='tanh').to(device)

# Generate random spin configurations
batch_size = 5
spins = torch.randint(0, 2, (batch_size, n_spins)).float() * 2 - 1  # Convert to +/- 1
spins = spins.to(device)

# Compute wave function
log_psi = fnn.log_psi(spins)
psi = fnn.psi(spins)

print("Spin configurations:")
print(spins.cpu().numpy())
print("\nLog of wave function:")
print(log_psi.cpu().detach().numpy())
print("\nWave function:")
print(psi.cpu().detach().numpy())

## 2.4 自回归神经网络量子态

自回归神经网络（Autoregressive Neural Network）是一种特殊类型的神经网络，它通过序列化地预测系统中的每个变量来构建联合概率分布。在量子态表示中，自回归神经网络可以高效地表示复杂的波函数。

### 自回归量子态的数学表示

对于自旋系统，自回归量子态可以表示为：

$$\psi(\mathbf{s}; \mathbf{\theta}) = \prod_{i=1}^{N} \psi_i(s_i | \mathbf{s}_{<i}; \mathbf{\theta})$$

其中$\mathbf{s}_{<i} = (s_1, s_2, ..., s_{i-1})$是前$i-1$个自旋的组态，$\psi_i(s_i | \mathbf{s}_{<i}; \mathbf{\theta})$是给定前$i-1$个自旋时第$i$个自旋的条件波函数。

每个条件波函数$\psi_i$可以表示为神经网络的输出：

$$\psi_i(s_i | \mathbf{s}_{<i}; \mathbf{\theta}) = f_{NN}^{(i)}(\mathbf{s}_{<i}; \mathbf{\theta})$$

其中$f_{NN}^{(i)}$是第$i$个神经网络的输出。

### 自回归量子态的特点

1. **高效采样**：可以直接从波函数中采样，无需马尔可夫链蒙特卡洛
2. **精确表示**：可以精确表示任意量子态（在足够大的网络容量下）
3. **条件独立性**：利用变量之间的条件独立性，减少参数数量
4. **序列化处理**：适合处理具有自然顺序的系统

In [ ]:
class AutoregressiveWavefunction(nn.Module):
    """
    Autoregressive Neural Network wave function for spin systems
    
    Parameters:
    n_visible: number of visible units (spins)
    hidden_size: size of hidden layer
    """
    def __init__(self, n_visible, hidden_size):
        super(AutoregressiveWavefunction, self).__init__()
        self.n_visible = n_visible
        self.hidden_size = hidden_size
        
        # Create a separate network for each spin
        self.networks = nn.ModuleList([
            nn.Sequential(
                nn.Linear(i, hidden_size),
                nn.Tanh(),
                nn.Linear(hidden_size, 2)  # Output for s_i = +1 and s_i = -1
            ) for i in range(n_visible)
        ])
        
    def forward(self, x):
        """
        Compute log of wave function amplitude
        
        Parameters:
        x: spin configurations, shape (batch_size, n_visible)
        
        Returns:
        log of wave function amplitude, shape (batch_size,)
        """
        batch_size = x.size(0)
        log_psi = torch.zeros(batch_size, device=x.device)
        
        for i in range(self.n_visible):
            # Get previous spins
            prev_spins = x[:, :i]
            
            # Get network output for current spin
            net_output = self.networks[i](prev_spins)
            
            # Get log probability for current spin value
            current_spins = x[:, i]
            spin_indices = (current_spins + 1) / 2  # Convert from +/-1 to 0/1
            spin_indices = spin_indices.long()
            
            # Use log_softmax for numerical stability
            log_probs = F.log_softmax(net_output, dim=1)
            selected_log_probs = log_probs[torch.arange(batch_size), spin_indices]
            
            # Add to total log probability
            log_psi += selected_log_probs
        
        return log_psi
    
    def log_psi(self, x):
        """
        Compute log of wave function
        
        Parameters:
        x: spin configurations, shape (batch_size, n_visible)
        
        Returns:
        log of wave function, shape (batch_size,)
        """
        return self.forward(x)
    
    def psi(self, x):
        """
        Compute wave function
        
        Parameters:
        x: spin configurations, shape (batch_size, n_visible)
        
        Returns:
        wave function, shape (batch_size,)
        """
        return torch.exp(self.log_psi(x))
    
    def sample(self, batch_size):
        """
        Sample from the wave function
        
        Parameters:
        batch_size: number of samples to generate
        
        Returns:
        samples: spin configurations, shape (batch_size, n_visible)
        """
        samples = torch.zeros(batch_size, self.n_visible, device=self.networks[0][0].weight.device)
        
        for i in range(self.n_visible):
            # Get previous spins
            prev_spins = samples[:, :i]
            
            # Get network output for current spin
            net_output = self.networks[i](prev_spins)
            
            # Get probabilities
            probs = F.softmax(net_output, dim=1)
            
            # Sample current spin
            spin_indices = torch.multinomial(probs, 1).squeeze(1)
            samples[:, i] = spin_indices * 2 - 1  # Convert from 0/1 to +/-1
        
        return samples
    
    def local_energy(self, x, hamiltonian):
        """
        Compute local energy
        
        Parameters:
        x: spin configurations, shape (batch_size, n_visible)
        hamiltonian: function that computes local energy
        
        Returns:
        local energy, shape (batch_size,)
        """
        return hamiltonian(x, self)

# Example: Create autoregressive wave function for a small spin system
n_spins = 4
hidden_size = 8
ar = AutoregressiveWavefunction(n_spins, hidden_size).to(device)

# Generate random spin configurations
batch_size = 5
spins = torch.randint(0, 2, (batch_size, n_spins)).float() * 2 - 1  # Convert to +/- 1
spins = spins.to(device)

# Compute wave function
log_psi = ar.log_psi(spins)
psi = ar.psi(spins)

print("Spin configurations:")
print(spins.cpu().numpy())
print("\nLog of wave function:")
print(log_psi.cpu().detach().numpy())
print("\nWave function:")
print(psi.cpu().detach().numpy())

# Sample from the wave function
samples = ar.sample(batch_size)
print("\nSamples from wave function:")
print(samples.cpu().detach().numpy())

## 2.5 卷积神经网络量子态

卷积神经网络（Convolutional Neural Network, CNN）是一种专门用于处理网格结构数据的神经网络。在量子态表示中，CNN特别适合处理具有空间局部性的系统，如晶格上的自旋系统。

### CNN量子态的数学表示

对于格点系统，CNN量子态可以表示为：

$$\psi(\mathbf{s}; \mathbf{\theta}) = e^{f_{CNN}(\mathbf{s}; \mathbf{\theta})}$$

其中$f_{CNN}(\mathbf{s}; \mathbf{\theta})$是卷积神经网络的输出，$\mathbf{\theta}$是网络的参数。

卷积神经网络通过卷积层、池化层和全连接层的组合来提取特征：

1. **卷积层**：应用卷积核提取局部特征
2. **池化层**：降低空间维度，减少参数数量
3. **全连接层**：将特征映射到最终输出

### CNN量子态的特点

1. **平移不变性**：CNN天然具有平移不变性，适合处理周期性边界条件
2. **局部性**：卷积操作捕捉局部关联，适合描述短程相互作用
3. **参数共享**：卷积核在整个输入上共享，减少了参数数量
4. **高效计算**：卷积操作可以利用GPU加速，适合大规模系统

In [ ]:
class CNNWavefunction(nn.Module):
    """
    Convolutional Neural Network (CNN) wave function for 2D spin systems
    
    Parameters:
    lattice_size: tuple (Lx, Ly) for lattice dimensions
    channels: list of channel sizes for each conv layer
    kernel_sizes: list of kernel sizes for each conv layer
    fc_sizes: list of fully connected layer sizes
    """
    def __init__(self, lattice_size, channels, kernel_sizes, fc_sizes):
        super(CNNWavefunction, self).__init__()
        self.lattice_size = lattice_size
        self.Lx, self.Ly = lattice_size
        self.n_spins = self.Lx * self.Ly
        
        # Create convolutional layers
        conv_layers = []
        in_channels = 1  # Input is a single channel (spin configuration)
        
        for out_channels, kernel_size in zip(channels, kernel_sizes):
            conv_layers.append(nn.Conv2d(in_channels, out_channels, kernel_size, padding=kernel_size//2))
            conv_layers.append(nn.Tanh())
            in_channels = out_channels
        
        self.conv_layers = nn.Sequential(*conv_layers)
        
        # Calculate size after convolutions
        self.conv_output_size = channels[-1] * self.Lx * self.Ly
        
        # Create fully connected layers
        fc_layers = []
        in_features = self.conv_output_size
        
        for out_features in fc_sizes:
            fc_layers.append(nn.Linear(in_features, out_features))
            fc_layers.append(nn.Tanh())
            in_features = out_features
        
        # Output layer (single neuron for log amplitude)
        fc_layers.append(nn.Linear(in_features, 1))
        
        self.fc_layers = nn.Sequential(*fc_layers)
        
    def forward(self, x):
        """
        Compute log of wave function amplitude
        
        Parameters:
        x: spin configurations, shape (batch_size, n_spins)
        
        Returns:
        log of wave function amplitude, shape (batch_size,)
        """
        batch_size = x.size(0)
        
        # Reshape to 2D lattice
        x = x.view(batch_size, 1, self.Lx, self.Ly)
        
        # Apply convolutional layers
        x = self.conv_layers(x)
        
        # Flatten
        x = x.view(batch_size, -1)
        
        # Apply fully connected layers
        x = self.fc_layers(x)
        
        return x.squeeze(1)  # Remove last dimension
    
    def log_psi(self, x):
        """
        Compute log of wave function
        
        Parameters:
        x: spin configurations, shape (batch_size, n_spins)
        
        Returns:
        log of wave function, shape (batch_size,)
        """
        return self.forward(x)
    
    def psi(self, x):
        """
        Compute wave function
        
        Parameters:
        x: spin configurations, shape (batch_size, n_spins)
        
        Returns:
        wave function, shape (batch_size,)
        """
        return torch.exp(self.log_psi(x))
    
    def local_energy(self, x, hamiltonian):
        """
        Compute local energy
        
        Parameters:
        x: spin configurations, shape (batch_size, n_spins)
        hamiltonian: function that computes local energy
        
        Returns:
        local energy, shape (batch_size,)
        """
        return hamiltonian(x, self)

# Example: Create CNN for a small 2D spin system
Lx, Ly = 2, 2  # 2x2 lattice
channels = [4, 8]
kernel_sizes = [3, 3]
fc_sizes = [16, 8]
cnn = CNNWavefunction((Lx, Ly), channels, kernel_sizes, fc_sizes).to(device)

# Generate random spin configurations
batch_size = 5
spins = torch.randint(0, 2, (batch_size, Lx*Ly)).float() * 2 - 1  # Convert to +/- 1
spins = spins.to(device)

# Compute wave function
log_psi = cnn.log_psi(spins)
psi = cnn.psi(spins)

print("Spin configurations:")
print(spins.cpu().numpy())
print("\nLog of wave function:")
print(log_psi.cpu().detach().numpy())
print("\nWave function:")
print(psi.cpu().detach().numpy())

# Visualize a spin configuration
sample_spin = spins[0].cpu().numpy().reshape(Lx, Ly)
plt.figure(figsize=(6, 6))
plt.imshow(sample_spin, cmap='RdBu', vmin=-1, vmax=1)
plt.colorbar()
plt.title('Sample Spin Configuration')
plt.xlabel('x')
plt.ylabel('y')
plt.show()

## 总结

本节介绍了各种类型的神经网络量子态，包括：

1. **神经网络作为波函数的表示**：介绍了神经网络量子态的基本原理和优势

2. **受限玻尔兹曼机（RBM）量子态**：详细介绍了RBM量子态的数学表示、特点和实现

3. **前馈神经网络量子态**：介绍了FNN量子态的数学表示、特点和实现

4. **自回归神经网络量子态**：介绍了自回归量子态的数学表示、特点和实现，强调了其高效采样的优势

5. **卷积神经网络量子态**：介绍了CNN量子态的数学表示、特点和实现，强调了其对空间局部性系统的适用性

这些不同类型的神经网络量子态各有特点，适用于不同的物理系统。在实际应用中，我们需要根据具体问题的特点选择合适的神经网络结构。